# Core Quickstart

This notebook is the lowest-level walkthrough of Agentic Forge. It defines a forge directly inside the notebook so you can see the abstraction before using the built-in components.

What you will do:

- define a custom `Material`
- define a custom `Forged` runtime object
- register a custom `Forge`
- load `materials/uppercase_material.yaml` through the public `forge(...)` entrypoint

The first code cell only resolves the repository root and imports the public API plus the core base classes used by the example.


In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents] if (p / 'agentic_forge').exists()), NOTEBOOK_DIR)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CORE_DIR = PROJECT_ROOT / 'examples' / 'core'

from agentic_forge import forge
from agentic_forge.core import Forge, Forged, Material, register_forge


## Step 1: Define and register the forge

This cell contains the three layers of the pattern:

- `UppercaseMaterial` is the declarative input loaded from YAML
- `ForgedText` is the runtime object returned at the end
- `UppercaseForge` is the builder that transforms the material into the runtime object

The important detail is that the forge is registered in the same way as the built-in components, using `@register_forge(...)`.


In [ ]:
class UppercaseMaterial(Material):
    prefix: str = ''
    text: str
    uppercase: bool = True


class ForgedText(Forged):
    def __init__(self, text: str) -> None:
        self.text = text

    def render(self) -> str:
        return self.text


@register_forge('example_uppercase')
class UppercaseForge(Forge):
    material_cls = UppercaseMaterial

    def forge_from_material(self, material: UppercaseMaterial) -> ForgedText:
        text = f'{material.prefix}{material.text}'
        if material.uppercase:
            text = text.upper()
        return ForgedText(text)


## Step 2: Inspect the YAML material

The file `materials/uppercase_material.yaml` is intentionally small so the mapping between YAML and Python stays obvious:

- `forge` selects `example_uppercase`
- `prefix`, `text`, and `uppercase` map directly to `UppercaseMaterial`

This is the core Agentic Forge idea: the YAML stays declarative while the Python layer holds the runtime behavior.


In [ ]:
material_path = CORE_DIR / 'materials' / 'uppercase_material.yaml'
print(material_path.read_text(encoding='utf-8'))


## Step 3: Forge the runtime object

Calling `forge(...)` loads the YAML, resolves the `forge` key through the registry, validates the material, and returns the final runtime object.

The output should show a `ForgedText` instance and the rendered string produced by `UppercaseForge`.


In [ ]:
result = forge(material_path)
print(type(result).__name__)
print(result.render())
